In [ ]:
!pip install kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download a standard landscape dataset (approx 7,000 images)
!kaggle datasets download -d theblackmamba31/landscape-image-colorization
!unzip -q landscape-image-colorization.zip -d landscape_data/

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/theblackmamba31/landscape-image-colorization
License(s): unknown
100% 192M/192M [00:05<00:00, 37.1MB/s]



In [ ]:
import tensorflow as tf
import os
import time

BATCH_SIZE = 8
IMG_SIZE = 256
EPOCHS = 10

# Path to the extracted color images
DATA_DIR = './landscape_data/landscape Images/color' # Adjust path based on the exact unzip structure

def process_path(file_path):
    # 1. Read and decode the image
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])

    # 2. Normalize the Color Target to [-1, 1]
    target_rgb = (img / 127.5) - 1.0

    # 3. Create the Grayscale Input dynamically (1-channel)
    # tf.image.rgb_to_grayscale uses standard luminosity weighting
    input_gray = tf.image.rgb_to_grayscale(img)
    input_gray = (input_gray / 127.5) - 1.0

    return input_gray, target_rgb

def build_landscape_dataset(data_dir):
    print("Building TensorFlow Data Pipeline...")
    # Get all image file paths
    file_paths = tf.data.Dataset.list_files(str(data_dir + '/*.jpg'))

    # Map the preprocessing function, shuffle, and batch
    dataset = file_paths.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.shuffle(buffer_size=1000)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

# Initialize the pipeline
train_dataset = build_landscape_dataset(DATA_DIR)

Building TensorFlow Data Pipeline...


In [ ]:
# ==========================================
# FAST-GAN GENERATOR (Keras 3 Compliant)
# ==========================================
def build_generator():
    inputs = tf.keras.Input(shape=[IMG_SIZE, IMG_SIZE, 1], name="Gray_Input")

    # ENCODER
    e1 = tf.keras.layers.Conv2D(64, kernel_size=4, strides=2, padding='same')(inputs)
    e1 = tf.keras.layers.LeakyReLU(negative_slope=0.2)(e1)

    # Feature Adaptation Module
    e2_dilated = tf.keras.layers.Conv2D(128, kernel_size=4, strides=1, padding='same', dilation_rate=2)(e1)
    e2_dilated = tf.keras.layers.BatchNormalization()(e2_dilated)
    e2_dilated = tf.keras.layers.LeakyReLU(negative_slope=0.2)(e2_dilated)

    e2 = tf.keras.layers.Conv2D(128, kernel_size=4, strides=2, padding='same')(e2_dilated)
    e2 = tf.keras.layers.BatchNormalization()(e2)
    e2 = tf.keras.layers.LeakyReLU(negative_slope=0.2)(e2)

    e3 = tf.keras.layers.Conv2D(256, kernel_size=4, strides=2, padding='same')(e2)
    e3 = tf.keras.layers.BatchNormalization()(e3)
    e3 = tf.keras.layers.LeakyReLU(negative_slope=0.2)(e3)

    # BOTTLENECK
    bottleneck = tf.keras.layers.Conv2D(512, kernel_size=4, strides=2, padding='same', activation='relu')(e3)

    # DECODER
    d1 = tf.keras.layers.Conv2DTranspose(256, kernel_size=4, strides=2, padding='same')(bottleneck)
    d1 = tf.keras.layers.BatchNormalization()(d1)
    d1 = tf.keras.layers.Activation('relu')(d1)

    # Stop-Gradient Semantic Fusion
    frozen_e3 = tf.keras.layers.Lambda(lambda x: tf.stop_gradient(x))(e3)
    d1_fused = tf.keras.layers.Concatenate()([d1, frozen_e3])

    d2 = tf.keras.layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding='same')(d1_fused)
    d2 = tf.keras.layers.BatchNormalization()(d2)
    d2 = tf.keras.layers.Activation('relu')(d2)

    d3 = tf.keras.layers.Conv2DTranspose(64, kernel_size=4, strides=2, padding='same')(d2)

    # SUPER-RESOLUTION REFINER (PixelShuffle)
    sr_prep = tf.keras.layers.Conv2D(12, kernel_size=3, padding='same', activation='relu')(d3)
    sr_out = tf.keras.layers.Lambda(lambda x: tf.nn.depth_to_space(x, block_size=2))(sr_prep)

    # Final Output Layer
    final_img = tf.keras.layers.Conv2D(3, kernel_size=4, strides=1, padding='same', activation='tanh')(sr_out)

    return tf.keras.Model(inputs=inputs, outputs=final_img)

# ==========================================
# PATCH-GAN DISCRIMINATOR
# ==========================================
def build_discriminator():
    inp_gray = tf.keras.Input(shape=[IMG_SIZE, IMG_SIZE, 1], name='gray_image')
    inp_rgb = tf.keras.Input(shape=[IMG_SIZE, IMG_SIZE, 3], name='rgb_image')

    concat = tf.keras.layers.Concatenate()([inp_gray, inp_rgb])

    d1 = tf.keras.layers.Conv2D(64, kernel_size=4, strides=2, padding='same')(concat)
    d1 = tf.keras.layers.LeakyReLU(negative_slope=0.2)(d1)

    d2 = tf.keras.layers.Conv2D(128, kernel_size=4, strides=2, padding='same')(d1)
    d2 = tf.keras.layers.BatchNormalization()(d2)
    d2 = tf.keras.layers.LeakyReLU(negative_slope=0.2)(d2)

    d3 = tf.keras.layers.Conv2D(256, kernel_size=4, strides=2, padding='same')(d2)
    d3 = tf.keras.layers.BatchNormalization()(d3)
    d3 = tf.keras.layers.LeakyReLU(negative_slope=0.2)(d3)

    patch_out = tf.keras.layers.Conv2D(1, kernel_size=4, strides=1, padding='same')(d3)
    return tf.keras.Model(inputs=[inp_gray, inp_rgb], outputs=patch_out)

generator = build_generator()
discriminator = build_discriminator()

# ==========================================
# LOSS, OPTIMIZERS & TRAINING
# ==========================================
loss_object = tf.keras.losses.BinaryCrossentropy(from_logits=True)
LAMBDA_L1 = 100

generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

@tf.function
def train_step(input_gray, target_rgb):
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        gen_output = generator(input_gray, training=True)

        disc_real_output = discriminator([input_gray, target_rgb], training=True)
        disc_generated_output = discriminator([input_gray, gen_output], training=True)

        # Generator Loss
        gan_loss = loss_object(tf.ones_like(disc_generated_output), disc_generated_output)
        l1_loss = tf.reduce_mean(tf.abs(target_rgb - gen_output))
        gen_loss = gan_loss + (LAMBDA_L1 * l1_loss)

        # Discriminator Loss
        real_loss = loss_object(tf.ones_like(disc_real_output), disc_real_output)
        generated_loss = loss_object(tf.zeros_like(disc_generated_output), disc_generated_output)
        disc_loss = real_loss + generated_loss

    generator_gradients = gen_tape.gradient(gen_loss, generator.trainable_variables)
    discriminator_gradients = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(generator_gradients, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(discriminator_gradients, discriminator.trainable_variables))

    return gen_loss, disc_loss

def fit(train_ds, epochs):
    print("Starting Landscape Colorization Training...\n")
    for epoch in range(epochs):
        start = time.time()
        print(f"Epoch {epoch+1}/{epochs} started...")

        for n, (input_gray, target_rgb) in train_ds.enumerate():
            gen_loss, disc_loss = train_step(input_gray, target_rgb)

            # Print every 10 batches to keep terminal clean
            if n % 10 == 0:
                print(f"  Batch {n} - Gen Loss: {gen_loss:.4f} | Disc Loss: {disc_loss:.4f}", end="\r")

        print(f"\nEpoch {epoch+1} completed in {time.time()-start:.2f} sec")

        # Save model weights every 5 epochs just in case
        if (epoch + 1) % 5 == 0:
            generator.save(f'fast_gan_landscape_epoch_{epoch+1}.keras')

# Execute Training
fit(train_dataset, EPOCHS)

Starting Landscape Colorization Training...

Epoch 1/10 started...


KeyboardInterrupt: 

In [1]:
from sentence_transformers import SentenceTransformer

In [2]:
model = SentenceTransformer("nomic-ai/nomic-embed-text-v1")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/71.3k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

In [3]:
model.save("nomic-ai")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [4]:
!zip /content/nomic-ai.zip /content/nomic-ai

  adding: content/nomic-ai/ (stored 0%)


In [5]:
import shutil
shutil.make_archive('nomic-ai', 'zip', 'nomic-ai')


'/content/nomic-ai.zip'